# Step 2e – Whisper-small (Step 2d-র সাথে তুলনার জন্য)

**Runtime → Change runtime type → L4 GPU** (বা T4) বেছে **Runtime → Run all** চাপুন।

- ডেটা, ভাগ, টেস্ট সেট আর ট্রেনিং সেটিং হুবহু Step 2d (turbo)-এর মতো, শুধু মডেল আলাদা, তাই দুটোর তুলনা ন্যায্য।

- ব্যবহৃত বক্তা: **A1, A2, A4, A6, A7**। ট্রেনিং: A1, A4, A6, A7। **A2** শুধু unseen test-এ।
- বাক্যের train/val/test ভাগ আগের মতোই (seed 13), তাই পরে নতুন রেকর্ডিং যোগ করলে একই টেস্ট সেটে তুলনা করা যাবে।
- প্রথমবার `QUICK = True` রেখে চালান (১৫–২০ মিনিট), সব ঠিক থাকলে `QUICK = False` করে আবার Run all।
- Checkpoint যায় Drive-এর `Sylheti_Thesis/sylheti_asr/whisper-small-4spk`-এ (প্রতিটা ~১.৫ GB, সর্বোচ্চ ২টা রাখা হয়)। Disconnect হলে আবার Run all দিলে শেষ checkpoint থেকে চলবে।

## 1. Drive, GPU আর library

In [ ]:
from google.colab import drive
import os
if not os.path.ismount('/content/drive') and os.path.isdir('/content/drive'):
    for d, _, _ in sorted(os.walk('/content/drive'), key=lambda x: -len(x[0])):
        try: os.rmdir(d)
        except OSError: pass
drive.mount('/content/drive')

In [ ]:
QUICK = False        # True = ছোট পরীক্ষা (~১৫–২০ মিনিট). False = পুরো training.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "transformers==4.48.3", "huggingface-hub", "accelerate==1.3.0",
                "jiwer==3.0.5", "soundfile", "bitsandbytes"], check=True)
import torch, shutil
assert torch.cuda.is_available(), "GPU নেই: Runtime -> Change runtime type -> L4 GPU"
GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
BF16 = torch.cuda.get_device_capability()[0] >= 8   # T4 -> fp16, L4 -> bf16
print(f"GPU: {GPU_NAME} ({GPU_GB:.0f} GB) | bf16: {BF16}")
print("Whisper-small T4 বা L4 দুটোতেই চলে।")

## 2. WAV ফিরিয়ে আনা (Drive-এর .tar থেকে)

In [ ]:
import tarfile, glob, re
import pandas as pd
ROOT = "/content/drive/MyDrive/Sylheti_Thesis/Ultimate Thisis Data"
SAVE = ROOT + "/wav_16k_tar"
OUT = "/content/wav_16k"
ASR_DIR = "/content/drive/MyDrive/Sylheti_Thesis/sylheti_asr"
TRAIN_SPK = ["A1", "A4", "A6", "A7"]
UNSEEN_SPK = ["A2"]
KEEP = TRAIN_SPK + UNSEEN_SPK                  # only these speakers are used anywhere
os.makedirs(OUT, exist_ok=True)

meta = pd.read_csv(ASR_DIR + "/metadata.csv")
meta = meta[meta.speaker.isin(KEEP)].copy()
meta["audio_path"] = OUT + "/" + meta.audio_path.str.split("/wav_16k/").str[-1]
meta["folder"] = meta.audio_path.str[len(OUT) + 1:].str.split("/").str[0]

for t in sorted(glob.glob(SAVE + "/*.tar")):
    s = os.path.basename(t)[:-4]
    if re.fullmatch(r"A\d+", s) and s not in KEEP:
        continue
    want = meta.audio_path[meta.folder == s]
    have_all = len(want) > 0 and want.map(os.path.exists).all()
    if have_all:
        continue
    print("extracting", s, "(", int((~want.map(os.path.exists)).sum()), "missing )")
    with tarfile.open(t) as tf:
        tf.extractall(OUT)                     # full re-extract repairs a half-extracted folder
print("wav ready:", int(meta.audio_path.map(os.path.exists).sum()), "/", len(meta))
u = shutil.disk_usage("/content/drive")
print(f"Drive: {u.free/1e9:.0f} GB খালি (দরকার ~৩–৪ GB)")

## 3. Train / validation / test

In [ ]:
ok = meta.audio_path.map(os.path.exists)
print("missing wav:", int((~ok).sum()))
if (~ok).any():
    print(meta[~ok].groupby(["speaker", "split"]).size().unstack(fill_value=0).to_string())
    print("উদাহরণ:", meta.audio_path[~ok].head(3).tolist())
print("৩০ সেকেন্ডের বেশি (বাদ):", int((meta.duration_sec > 30).sum()))
meta = meta[ok & (meta.duration_sec <= 30) & meta.text.notna()
            & meta.split.isin(["train", "validation", "test_seen", "test_unseen"])].reset_index(drop=True)
assert set(meta.speaker) <= set(KEEP)
assert set(meta[meta.split != "test_unseen"].speaker) <= set(TRAIN_SPK)
assert set(meta[meta.split == "test_unseen"].speaker) <= set(UNSEEN_SPK)

summary = meta.groupby(["speaker", "split"]).agg(n=("audio_path", "size"), hours=("duration_sec", lambda x: round(x.sum() / 3600, 2)))
print(summary.n.unstack(fill_value=0).to_string())
print("\nঘণ্টা:"); print(meta.groupby("split").duration_sec.sum().div(3600).round(2).to_string())

sp = {k: g.reset_index(drop=True) for k, g in meta.groupby("split")}
train_df, val_df = sp["train"], sp["validation"]
val_small = val_df.sample(min(300, len(val_df)), random_state=13).reset_index(drop=True)
if QUICK:
    train_df = train_df.sample(400, random_state=13).reset_index(drop=True)
    val_small = val_small.head(50)
print("\ntrain:", len(train_df), "| validation used during training:", len(val_small))

## 4. Model আর data loader

In [ ]:
import re, unicodedata, numpy as np, soundfile as sf, jiwer
from transformers import WhisperProcessor, WhisperForConditionalGeneration
BASE = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(BASE, language="bengali", task="transcribe")
tok, fe = processor.tokenizer, processor.feature_extractor

ZW = dict.fromkeys(map(ord, "​‌﻿­⁠"), None)
def clean(s):
    """Same normalisation as Step 2b."""
    s = unicodedata.normalize("NFC", str(s)).translate(ZW)
    s = re.sub("‍(?!্য)", "", s)
    s = re.sub(r"[\"\'“”‘’(),.।?!:;\-/*…\[\]|]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

class SpeechSet(torch.utils.data.Dataset):
    def __init__(self, df): self.df = df
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        a, sr = sf.read(r.audio_path, dtype="float32")
        if a.ndim > 1: a = a.mean(1)
        feats = fe(a[: 30 * 16000], sampling_rate=16000).input_features[0]
        labels = tok(clean(r.text)).input_ids[:448]
        return {"input_features": feats, "labels": labels}

def collate(batch):
    x = fe.pad([{"input_features": b["input_features"]} for b in batch], return_tensors="pt")
    y = tok.pad([{"input_ids": b["labels"]} for b in batch], return_tensors="pt")
    labels = y.input_ids.masked_fill(y.attention_mask.ne(1), -100)
    if (labels[:, 0] == model.config.decoder_start_token_id).all():
        labels = labels[:, 1:]
    x["labels"] = labels
    return x

def setup(m):
    m.generation_config.language = "bengali"
    m.generation_config.task = "transcribe"
    m.generation_config.forced_decoder_ids = None
    m.config.forced_decoder_ids = None
    return m

model = setup(WhisperForConditionalGeneration.from_pretrained(BASE))
model.config.apply_spec_augment = True
model.config.mask_time_prob = 0.05
model.config.use_cache = False

def compute_metrics(p):
    pred, lab = p.predictions, p.label_ids
    lab = np.where(lab == -100, tok.pad_token_id, lab)
    hyp = [clean(t) for t in tok.batch_decode(pred, skip_special_tokens=True)]
    ref = [clean(t) for t in tok.batch_decode(lab, skip_special_tokens=True)]
    pairs = [(r, h) for r, h in zip(ref, hyp) if r]
    ref, hyp = [r for r, _ in pairs], [h if h else "-" for _, h in pairs]
    return {"wer": 100 * jiwer.wer(ref, hyp), "cer": 100 * jiwer.cer(ref, hyp)}
print("ready:", BASE, f"({sum(p.numel() for p in model.parameters())/1e6:.0f}M params)")

## 5. Training (checkpoint Drive-এ, resume হয়)

In [ ]:
# TRAIN = True  -> train / resume from the last checkpoint
# TRAIN = False -> skip training, use the saved final model
TRAIN = True
import json, gc, os, glob
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallback
from transformers.trainer_utils import get_last_checkpoint
RUN = ASR_DIR + ("/whisper-small-4spk-quick" if QUICK else "/whisper-small-4spk")
FINAL = RUN + "/final"
os.makedirs(RUN, exist_ok=True)

class StopIfNoBetterWER(TrainerCallback):
    """Stop when validation WER has not reached a new best for `wait` steps (same rule as Step 2b)."""
    def __init__(self, wait): self.wait = wait
    def on_evaluate(self, args, state, control, metrics=None, **kw):
        wer = (metrics or {}).get("eval_wer")
        if wer is None or state.best_metric is None or wer < state.best_metric or not state.best_model_checkpoint:
            return
        best_step = int(str(state.best_model_checkpoint).rstrip("/").split("-")[-1])
        if state.global_step - best_step >= self.wait:
            print(f"No new best WER for {self.wait} steps (best {state.best_metric:.2f} at {best_step}) -> stopping")
            control.should_training_stop = True

BIG = GPU_GB >= 20
if TRAIN:
    args = Seq2SeqTrainingArguments(
        output_dir=RUN,
        per_device_train_batch_size=16, gradient_accumulation_steps=2,   # 32 clips per update, as in Step 2d
        per_device_eval_batch_size=16,
        learning_rate=1e-5, warmup_steps=20 if QUICK else 300,
        num_train_epochs=5, max_steps=20 if QUICK else -1,
        bf16=BF16, fp16=not BF16,
        optim="adamw_bnb_8bit",                               # same optimiser as Step 2d (fair comparison)
        gradient_checkpointing=True,
        eval_strategy="steps", save_strategy="steps",
        eval_steps=10 if QUICK else 250, save_steps=10 if QUICK else 250,
        save_total_limit=2, load_best_model_at_end=True,
        metric_for_best_model="wer", greater_is_better=False,
        predict_with_generate=True, generation_max_length=225,
        logging_steps=5 if QUICK else 50, report_to="none",
        dataloader_num_workers=2, remove_unused_columns=False,
    )
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=SpeechSet(train_df), eval_dataset=SpeechSet(val_small),
                             data_collator=collate, compute_metrics=compute_metrics,
                             callbacks=[StopIfNoBetterWER(wait=20 if QUICK else 1000)])

    # a checkpoint whose save was cut off (no trainer_state.json) is renamed, not deleted,
    # so training resumes from the last COMPLETE checkpoint instead of from the start
    for d in glob.glob(RUN + "/checkpoint-*"):
        if os.path.exists(d + "/trainer_state.json"):
            print("OK checkpoint     :", os.path.basename(d))
        else:
            new = d.replace("/checkpoint-", "/incomplete-checkpoint-")
            os.rename(d, new)
            print("renamed incomplete:", os.path.basename(d), "->", os.path.basename(new))

    last = get_last_checkpoint(RUN)
    print("resuming from", last if last else "the start")
    trainer.train(resume_from_checkpoint=last)
    best = trainer.state.best_model_checkpoint
    print("best checkpoint:", best, "| best validation WER:", trainer.state.best_metric)
    trainer.save_model(FINAL); processor.save_pretrained(FINAL)
    json.dump({"best_checkpoint": best, "best_val_wer": trainer.state.best_metric,
               "steps": trainer.state.global_step, "epochs": round(trainer.state.epoch, 2),
               "train_clips": len(train_df), "train_speakers": TRAIN_SPK, "unseen_speakers": UNSEEN_SPK},
              open(RUN + "/train_info.json", "w"), indent=1)
    del trainer
print("final model ->", FINAL)
gc.collect(); torch.cuda.empty_cache()

## 6. Test (Step 2d-র একই টেস্ট সেট)

In [ ]:
# ZERO_SHOT = True -> also test the untrained Whisper-small (optional; the turbo zero-shot is in Step 2d)
ZERO_SHOT = False
RERUN_TEST = False
from tqdm.auto import tqdm
DT = torch.bfloat16 if BF16 else torch.float16

def transcribe(m, df, bs=16):
    m.eval(); m.config.use_cache = True; out = []
    for i in tqdm(range(0, len(df), bs)):
        b = [SpeechSet(df.iloc[i:i + bs].reset_index(drop=True))[j] for j in range(min(bs, len(df) - i))]
        x = fe.pad([{"input_features": e["input_features"]} for e in b], return_tensors="pt")
        with torch.no_grad(), torch.autocast("cuda", dtype=DT):
            ids = m.generate(x.input_features.to("cuda"), max_new_tokens=440)
        out += tok.batch_decode(ids, skip_special_tokens=True)
    return out

def score(df):
    ref, hyp = df.text.map(clean).tolist(), [clean(h) or "-" for h in df.pred]
    return round(100 * jiwer.wer(ref, hyp), 1), round(100 * jiwer.cer(ref, hyp), 1)

RES = RUN + "/results.csv"
if os.path.exists(RES) and not RERUN_TEST:
    print("Saved results (RERUN_TEST = True দিলে আবার টেস্ট হবে):")
    res = pd.read_csv(RES)
else:
    systems = {"whisper-small fine-tuned (A1,A4,A6,A7)": FINAL}
    if ZERO_SHOT: systems["whisper-small zero-shot"] = BASE
    rows, per_spk = [], []
    for sys_name, path in systems.items():
        del model; gc.collect(); torch.cuda.empty_cache()
        model = setup(WhisperForConditionalGeneration.from_pretrained(path, torch_dtype=DT)).to("cuda")
        tag = "ft" if path == FINAL else "zs"
        for name in ["test_seen", "test_unseen"]:
            df = sp[name].head(100) if QUICK else sp[name]
            df = df.assign(pred=transcribe(model, df))
            w, c = score(df)
            rows.append({"model": sys_name, "test set": name, "files": len(df), "WER": w, "CER": c})
            for s, g in df.groupby("speaker"):
                per_spk.append({"model": sys_name, "test set": name, "speaker": s, "files": len(g), **dict(zip(["WER", "CER"], score(g)))})
            df[["audio_path", "speaker", "excel_row", "text", "pred"]].to_csv(f"{RUN}/predictions_{tag}_{name}.csv", index=False, encoding="utf-8-sig")
    res = pd.DataFrame(rows); res.to_csv(RES, index=False)
    pd.DataFrame(per_spk).to_csv(RUN + "/results_per_speaker.csv", index=False)
print(res.to_string(index=False))
if os.path.exists(RUN + "/results_per_speaker.csv"):
    print("\nবক্তা অনুযায়ী:"); print(pd.read_csv(RUN + "/results_per_speaker.csv").to_string(index=False))
print("\nsaved:", RUN)

## 7. কয়েকটা উদাহরণ (A2, অদেখা বক্তা)

In [ ]:
from IPython.display import Audio, display
d = pd.read_csv(f"{RUN}/predictions_ft_test_unseen.csv").sample(5, random_state=1)
for _, r in d.iterrows():
    print("আসল   :", r.text); print("মডেল  :", r.pred); display(Audio(r.audio_path)); print()

---
`QUICK = True`-এ শেষ পর্যন্ত error ছাড়া চললে উপরে `QUICK = False` করে আবার **Run all** দিন।
পুরো training L4-এ আনুমানিক ১–২ ঘণ্টা (~১০–১৫ ইউনিট), T4-এ ৩–৪ ঘণ্টা। শেষ হলে **Runtime → Disconnect and delete runtime** দিন, না হলে ইউনিট কাটতে থাকবে।